# 12. Final Results Synthesis and Experimental Analysis

**Project:** Uncertainty-Aware Terrain Semantic Segmentation and Traversability Estimation for Autonomous Planetary Rover Navigation  
**Author / Repository:** IpsitaHQ / [UA-MTSeg-Rover-Navigation](https://github.com/IpsitaHQ/UA-MTSeg-Rover-Navigation)  
**Notebook Scope:** Final results synthesis, statistical cross-verification, and publication-ready reporting across all completed research stages.

---

### Important Scientific Scope and Evaluated Framework Constraints

To preserve strict scientific integrity, the findings presented in this notebook adhere to the following explicit constraints:
1. **Validation Set Scope:** All segmentation, uncertainty, and planning evaluations are strictly conducted on the **190 validation images** ($12,451,840$ total pixels) from the BASEPROD 80/20 split. No independent held-out Martian test set is available or claimed.
2. **Offline Image-Space Formulation:** The $47.37\%$ path reachability metric is strictly an **offline image-space path-planning success rate** on fixed diagonal coordinates $(5, 5) 	o (250, 250)$. It must **not** be termed "navigation accuracy".
3. **No Physical Rover Experiments:** No physical planetary rover, field testbed, or analog site deployment was performed.
4. **No 3D Terrain or Vehicle Dynamics:** The planning proxy does not model digital elevation models (DEM), slope geometry, slip, ground clearance, tip-over stability, or rover kinodynamics.
5. **No Supervised Ground-Truth Traversability:** Traversability costs are engineered heuristic proxies mapped from semantic class IDs and normalized Shannon entropy.
6. **Predictive Uncertainty Formulation:** Predictive uncertainty is computed via single-pass normalized Shannon entropy over softmax probabilities, capturing combined aleatoric and model uncertainty rather than decomposed epistemic uncertainty (such as MC Dropout or deep ensembles).
7. **No Pareto Optimality Claims:** Detour-uncertainty trade-offs reflect parameter sensitivity within the evaluated proxy formulation, not formal Pareto optimality or definitive mathematical proof.
8. **Documentation of Missing Baseline Data:** While overall validation mIoU ($0.2862$) and pixel accuracy ($0.6687$) were recorded for the 5-epoch RGB baseline (Notebook 05), per-class IoU values were not saved in the source baseline outputs. In accordance with scientific reporting standards, these values are designated as *Not Recorded in Source Baseline* rather than retrospectively simulated or invented.
9. **Modality Alignment:** Depth and thermal channels are paired from the BASEPROD dataset using identical timestamp keys and resized to uniform resolution ($256 	imes 256$). Sub-pixel geometric calibration was not independently verified by internal sensor calibration routines in this codebase.
10. **Training Duration Disparity:** The reference RGB baseline was trained for 5 epochs (yielding validation mIoU $0.2862$, pixel accuracy $0.6687$), whereas the multimodal early-fusion model was trained for 25 epochs (reaching its best validation checkpoint at epoch 24 with mIoU $0.6462$, pixel accuracy $0.8609$). The observed gains reflect both the addition of depth and thermal modalities and extended training optimization duration.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 1 else f'{x:.2f}')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

OUTPUTS_DIR = ROOT / 'outputs'
FIGURES_DIR = ROOT / 'figures' / 'final_synthesis'
if not FIGURES_DIR.is_dir() and (OUTPUTS_DIR / 'final_synthesis').is_dir():
    FIGURES_DIR = OUTPUTS_DIR / 'final_synthesis'

print(f"Project Root: {ROOT}")
print(f"Figures Dir:  {FIGURES_DIR}")
print(f"Outputs Dir:  {OUTPUTS_DIR}")


---
## 1. RGB Baseline vs. RGB-D-T Multimodal Terrain Segmentation

Comparison of the reference RGB-only U-Net baseline (Notebook 05, 5 epochs) against the early-fusion multimodal U-Net (Notebook 08 / Epoch 24 checkpoint, 5 input channels: RGB + Depth + Thermal, 25 epochs).

- **Frozen Multimodal Model:** `outputs/multimodal_segmentation/best_multimodal_unet.pt` (Epoch 24)
- **Input Channels:** 5 (RGB + metric Depth + Thermal Celsius)
- **Parameters:** 7,766,792
- **Image Resolution:** $256 	imes 256$ pixels
- **Split:** 760 training samples, 190 validation samples (Seed 42)

*Note on Training Duration Disparity:* The RGB baseline was trained for 5 epochs, while the multimodal model was trained for 25 epochs (with the best checkpoint selected at epoch 24). Performance differences reflect both the integration of multimodal sensory data (depth + thermal) and extended optimization duration.


In [ ]:
# Table 1: Model Segmentation Performance Comparison
table1_data = [
    {
        "Model": "RGB Baseline U-Net",
        "Input Modalities": "3 ch (RGB)",
        "Epochs": 5,
        "Val mIoU": 0.2862,
        "Val Pixel Accuracy": 0.668699,
        "Absolute Gain (mIoU)": "Baseline",
        "Percentage Point Gain (mIoU)": "Baseline",
        "Relative Improvement (mIoU)": "Baseline",
        "Relative Improvement (Acc)": "Baseline"
    },
    {
        "Model": "Multimodal U-Net (Early Fusion)",
        "Input Modalities": "5 ch (RGB + Depth + Thermal)",
        "Epochs": 25,
        "Val mIoU": 0.6462015,
        "Val Pixel Accuracy": 0.8609459,
        "Absolute Gain (mIoU)": "+0.3600",
        "Percentage Point Gain (mIoU)": "+36.00 pp",
        "Relative Improvement (mIoU)": "+125.79%",
        "Relative Improvement (Acc)": "+28.75%"
    }
]

df_table1 = pd.DataFrame(table1_data)
display(df_table1)


In [ ]:
# Display Figure 1: Segmentation Comparison
f1_path = FIGURES_DIR / 'fig1_segmentation_comparison.png'
if f1_path.is_file():
    display(Image.open(f1_path))


---
## 2. Per-Class Intersection-over-Union (IoU) Breakdown

The BASEPROD dataset classifies planetary analog terrain into eight mutually exclusive classes.
The table below documents per-class IoU, validation pixel abundance, and observed error rates for each terrain category.

*Note on Baseline Data:* Per-class IoU metrics for the 5-epoch RGB baseline were not logged in the original baseline outputs; they are marked as *Not recorded in source baseline* to preserve archival documentation integrity.


In [ ]:
# Table 2: Per-Class Segmentation Performance (190 Validation Frames)
table2_data = [
    {"Class ID": 0, "Terrain Class": "Void", "Validation Pixels": 1044531, "Pixel Fraction (%)": 8.39, "Error Rate (%)": 24.01, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.667920},
    {"Class ID": 1, "Terrain Class": "Compact", "Validation Pixels": 6035201, "Pixel Fraction (%)": 48.47, "Error Rate (%)": 6.27, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.847922},
    {"Class ID": 2, "Terrain Class": "Grass", "Validation Pixels": 847749, "Pixel Fraction (%)": 6.81, "Error Rate (%)": 29.82, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.514720},
    {"Class ID": 3, "Terrain Class": "Bedrock", "Validation Pixels": 153052, "Pixel Fraction (%)": 1.23, "Error Rate (%)": 32.45, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.577139},
    {"Class ID": 4, "Terrain Class": "Sandy", "Validation Pixels": 1101710, "Pixel Fraction (%)": 8.85, "Error Rate (%)": 10.06, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.767217},
    {"Class ID": 5, "Terrain Class": "Pebble / Gravel", "Validation Pixels": 2475015, "Pixel Fraction (%)": 19.88, "Error Rate (%)": 17.13, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.749426},
    {"Class ID": 6, "Terrain Class": "Rock", "Validation Pixels": 223683, "Pixel Fraction (%)": 1.80, "Error Rate (%)": 43.64, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.454099},
    {"Class ID": 7, "Terrain Class": "Vegetation / Bush", "Validation Pixels": 570899, "Pixel Fraction (%)": 4.58, "Error Rate (%)": 29.31, "RGB Baseline IoU": "Not recorded", "Multimodal IoU": 0.591169},
]

df_table2 = pd.DataFrame(table2_data)
display(df_table2)
print(f"Validation Mean IoU across all 8 classes: {df_table2['Multimodal IoU'].mean():.4f}")


In [ ]:
# Display Figure 2: Per-Class IoU Breakdown
f2_path = FIGURES_DIR / 'fig2_per_class_iou.png'
if f2_path.is_file():
    display(Image.open(f2_path))


---
## 3. Predictive Entropy and Segmentation Error Detection Analysis

In Stage A (Notebook 09), normalized Shannon entropy was evaluated across all $12,451,840$ validation pixels:
$$H_{\text{norm}}(p) = -\frac{\sum_{c=1}^8 p_c \ln(p_c + 10^{-8})}{\ln(8)}$$

### Key Statistical Separation:
- **Correct Pixels:** Mean entropy = $0.1977$, Median entropy = $0.1324$ ($\pm 0.1870$)
- **Incorrect Pixels:** Mean entropy = $0.5022$, Median entropy = $0.5167$ ($\pm 0.1864$)
- **Entropy Separation Ratio:** **$2.54\times$ higher mean entropy in erroneous pixels**
- **AUROC for Error Detection:** **$0.8675$** (Area Under the Receiver Operating Characteristic curve)


In [ ]:
# Table 3: Binned Predictive Entropy and Segmentation Error Detection Breakdown
table3_data = [
    {"Entropy Bin Range": "0.0 - 0.2", "Pixel Count": 6858779, "Pixel Share (%)": 55.08, "Error Pixels": 125809, "Empirical Error Rate (%)": 1.83, "Mean Bin Entropy": 0.0788},
    {"Entropy Bin Range": "0.2 - 0.4", "Pixel Count": 2615826, "Pixel Share (%)": 21.01, "Error Pixels": 368027, "Empirical Error Rate (%)": 14.07, "Mean Bin Entropy": 0.2907},
    {"Entropy Bin Range": "0.4 - 0.6", "Pixel Count": 1914927, "Pixel Share (%)": 15.38, "Error Pixels": 672827, "Empirical Error Rate (%)": 35.14, "Mean Bin Entropy": 0.4954},
    {"Entropy Bin Range": "0.6 - 0.8", "Pixel Count": 942811, "Pixel Share (%)": 7.57, "Error Pixels": 488426, "Empirical Error Rate (%)": 51.81, "Mean Bin Entropy": 0.6776},
    {"Entropy Bin Range": "0.8 - 1.0", "Pixel Count": 119497, "Pixel Share (%)": 0.96, "Error Pixels": 76443, "Empirical Error Rate (%)": 63.97, "Mean Bin Entropy": 0.8436},
]

df_table3 = pd.DataFrame(table3_data)
display(df_table3)


In [ ]:
# Display Figure 3: Entropy vs Error Rate
f3_path = FIGURES_DIR / 'fig3_entropy_vs_error_rate.png'
if f3_path.is_file():
    display(Image.open(f3_path))


---
## 4. Offline Image-Space Path Planning: Semantic Baseline vs. 0.7/0.3 Uncertainty-Aware Formulation

In Stage B (Notebook 10), an offline 8-connected image-space $A^*$ search evaluated the cost of path safety:
- **Method A (Semantic Baseline):** $C_A = C_s$
- **Method B (Uncertainty-Aware):** $C_B = \min(239, \text{round}(0.7 C_s + 0.3 C_u))$ for $C_s < 240$; $C_B = 255$ for $C_s \ge 240$.
- **Fixed Endpoints:** Start $(5, 5)$, Goal $(250, 250)$ across all 190 validation frames.


In [ ]:
# Table 4: Paired Path Planning Comparison (Stage B, 190 Validation Frames)
table4_data = [
    {
        "Metric": "Total Validation Frames Evaluated",
        "Method A: Semantic Baseline": "190",
        "Method B: Uncertainty-Aware (0.7/0.3)": "190",
        "Paired Difference / Relative Delta": "Identical protocol"
    },
    {
        "Metric": "Successful Routes (Reachable)",
        "Method A: Semantic Baseline": "90",
        "Method B: Uncertainty-Aware (0.7/0.3)": "90",
        "Paired Difference / Relative Delta": "0 net change"
    },
    {
        "Metric": "Failed Routes (Unreachable Endpoints)",
        "Method A: Semantic Baseline": "100",
        "Method B: Uncertainty-Aware (0.7/0.3)": "100",
        "Paired Difference / Relative Delta": "100% agreement"
    },
    {
        "Metric": "Offline Path-Planning Success Rate",
        "Method A: Semantic Baseline": "47.37%",
        "Method B: Uncertainty-Aware (0.7/0.3)": "47.37%",
        "Paired Difference / Relative Delta": "0.0 pp difference"
    },
    {
        "Metric": "Mean Path Length (pixels)",
        "Method A: Semantic Baseline": "384.81 px",
        "Method B: Uncertainty-Aware (0.7/0.3)": "389.06 px",
        "Paired Difference / Relative Delta": "+4.25 px (+1.11% overhead)"
    },
    {
        "Metric": "Median Path Length (pixels)",
        "Method A: Semantic Baseline": "380.75 px",
        "Method B: Uncertainty-Aware (0.7/0.3)": "385.09 px",
        "Paired Difference / Relative Delta": "+4.34 px (+1.14% overhead)"
    },
    {
        "Metric": "Mean Path Cost (Cumulative)",
        "Method A: Semantic Baseline": "21,560.05",
        "Method B: Uncertainty-Aware (0.7/0.3)": "20,628.43",
        "Paired Difference / Relative Delta": "-4.32% cost"
    },
    {
        "Metric": "Median Path Cost (Cumulative)",
        "Method A: Semantic Baseline": "20,433.92",
        "Method B: Uncertainty-Aware (0.7/0.3)": "19,182.54",
        "Paired Difference / Relative Delta": "-6.12% cost"
    },
    {
        "Metric": "Mean Route Uncertainty (H_norm)",
        "Method A: Semantic Baseline": "0.2723",
        "Method B: Uncertainty-Aware (0.7/0.3)": "0.1643",
        "Paired Difference / Relative Delta": "-0.1081 (-39.68% reduction)"
    },
    {
        "Metric": "Median Route Uncertainty (H_norm)",
        "Method A: Semantic Baseline": "0.2650",
        "Method B: Uncertainty-Aware (0.7/0.3)": "0.1491",
        "Paired Difference / Relative Delta": "-0.1159 (-43.74% reduction)"
    },
    {
        "Metric": "Paths with Reduced Uncertainty",
        "Method A: Semantic Baseline": "N/A",
        "Method B: Uncertainty-Aware (0.7/0.3)": "90 / 90 (100.0%)",
        "Paired Difference / Relative Delta": "100.0% of reachable routes"
    },
    {
        "Metric": "Paths with Altered Trajectories",
        "Method A: Semantic Baseline": "N/A",
        "Method B: Uncertainty-Aware (0.7/0.3)": "90 / 90 (100.0%)",
        "Paired Difference / Relative Delta": "100.0% topological change"
    }
]

df_table4 = pd.DataFrame(table4_data)
display(df_table4)


In [ ]:
# Display Figure 4: Planning Risk-Distance Trade-Off
f4_path = FIGURES_DIR / 'fig4_planning_risk_distance_tradeoff.png'
if f4_path.is_file():
    display(Image.open(f4_path))


---
## 5. Parameter Sensitivity: Uncertainty Weight Ablation ($w_u \in [0.0, 0.5]$)

In Stage C (Notebook 11), the sensitivity of the planning result to the choice of uncertainty weight was evaluated across four configurations:
1. **Semantic Baseline:** $w_s=1.0, w_u=0.0$
2. **Configuration 1 (Low):** $w_s=0.9, w_u=0.1$
3. **Configuration 2 (Stage B Primary):** $w_s=0.7, w_u=0.3$
4. **Configuration 3 (High):** $w_s=0.5, w_u=0.5$


In [ ]:
# Table 5: Weight Sensitivity Analysis Across 190 Validation Frames
table5_data = [
    {
        "Configuration": "Semantic Baseline",
        "Semantic Weight (ws)": 1.0,
        "Uncertainty Weight (wu)": 0.0,
        "Success Rate (%)": 47.37,
        "Mean Path Length (px)": 384.81,
        "Mean Route Uncertainty": 0.2723,
        "Uncertainty Reduction (%)": 0.00,
        "Length Overhead (%)": 0.00,
        "Paths Changed vs Baseline": "0 / 90 (0.0%)"
    },
    {
        "Configuration": "Configuration 1 (Low)",
        "Semantic Weight (ws)": 0.9,
        "Uncertainty Weight (wu)": 0.1,
        "Success Rate (%)": 47.37,
        "Mean Path Length (px)": 386.49,
        "Mean Route Uncertainty": 0.1929,
        "Uncertainty Reduction (%)": 29.19,
        "Length Overhead (%)": 0.44,
        "Paths Changed vs Baseline": "86 / 90 (95.6%)"
    },
    {
        "Configuration": "Configuration 2 (Stage B)",
        "Semantic Weight (ws)": 0.7,
        "Uncertainty Weight (wu)": 0.3,
        "Success Rate (%)": 47.37,
        "Mean Path Length (px)": 389.06,
        "Mean Route Uncertainty": 0.1643,
        "Uncertainty Reduction (%)": 39.68,
        "Length Overhead (%)": 1.11,
        "Paths Changed vs Baseline": "90 / 90 (100.0%)"
    },
    {
        "Configuration": "Configuration 3 (High)",
        "Semantic Weight (ws)": 0.5,
        "Uncertainty Weight (wu)": 0.5,
        "Success Rate (%)": 47.37,
        "Mean Path Length (px)": 396.00,
        "Mean Route Uncertainty": 0.1443,
        "Uncertainty Reduction (%)": 47.02,
        "Length Overhead (%)": 2.91,
        "Paths Changed vs Baseline": "90 / 90 (100.0%)"
    }
]

df_table5 = pd.DataFrame(table5_data)
display(df_table5)


In [ ]:
# Display Figure 5: Weight Sensitivity Trade-Off
f5_path = FIGURES_DIR / 'fig5_weight_sensitivity_curves.png'
if f5_path.is_file():
    display(Image.open(f5_path))


---
## 6. Qualitative Visual Analysis: Representative Path Trajectories

The composite below displays paired planning trajectories on representative validation frames, illustrating how uncertainty cost guides paths around high-entropy terrain transition boundaries while semantic obstacles remain strictly respected.


In [ ]:
# Display Figure 6: Representative Paths Mosaic
f6_path = FIGURES_DIR / 'fig6_representative_paths_mosaic.png'
if f6_path.is_file():
    display(Image.open(f6_path))


---
## 7. Consolidated Executive Summary of Research Findings

This master table synthesizes the empirical evidence established across all experimental stages of the investigation.


In [ ]:
# Table 6: Consolidated Master Findings Table
table6_data = [
    {
        "Research Dimension": "1. Multimodal Early Fusion",
        "Evaluated Baseline / Control": "RGB-only U-Net (3 ch, 5 ep): mIoU = 0.2862, Acc = 0.6687",
        "Proposed System": "Multimodal U-Net (5 ch: RGB-D-T, 25 ep, best ep 24): mIoU = 0.6462, Acc = 0.8609",
        "Observed Quantitative Impact": "+0.3600 mIoU (+36.00 pp, +125.8% rel.); +0.1922 Acc (+19.22 pp, +28.8% rel.)",
        "Scientific Conclusion / Scope": "Early fusion of thermal and depth data alongside extended training substantially improves terrain classification on the BASEPROD validation set."
    },
    {
        "Research Dimension": "2. Predictive Entropy and Error Detection",
        "Evaluated Baseline / Control": "Correct pixel entropy: Mean = 0.1977, Median = 0.1324",
        "Proposed System": "Incorrect pixel entropy: Mean = 0.5022, Median = 0.5167",
        "Observed Quantitative Impact": "2.54× higher mean entropy in erroneous pixels; AUROC = 0.8675 for error detection",
        "Scientific Conclusion / Scope": "Predictive Shannon entropy is strongly correlated with segmentation failure on the 190 validation frames."
    },
    {
        "Research Dimension": "3. Risk-Aware Path Planning (Stage B)",
        "Evaluated Baseline / Control": "Semantic-only A* (CA = Cs): Mean U = 0.2723, Mean Len = 384.81 px",
        "Proposed System": "Uncertainty-Aware A* (CB = 0.7Cs + 0.3Cu): Mean U = 0.1643, Mean Len = 389.06 px",
        "Observed Quantitative Impact": "-39.68% mean route uncertainty at +1.11% length overhead (+4.25 px); 100% routes rerouted",
        "Scientific Conclusion / Scope": "Uncertainty cost routes paths through high-confidence semantic regions at minimal 2D geometric detour costs."
    },
    {
        "Research Dimension": "4. Weight Sensitivity (Stage C)",
        "Evaluated Baseline / Control": "Fixed weight wu = 0.3 (Stage B primary formulation)",
        "Proposed System": "Ablation over wu in [0.0, 0.1, 0.3, 0.5]",
        "Observed Quantitative Impact": "Monotonic trade-off: wu=0.1 (-29.2% u, +0.44% len); wu=0.5 (-47.0% u, +2.91% len)",
        "Scientific Conclusion / Scope": "Uncertainty reduction is robust and monotonic across weights; wu = 0.3 provides a balanced operating point."
    },
    {
        "Research Dimension": "5. Path Reachability / Obstacle Guarantee",
        "Evaluated Baseline / Control": "Semantic baseline reachability = 90 / 190 (47.37%)",
        "Proposed System": "Uncertainty-aware reachability across all wu in [0.1, 0.5] = 90 / 190 (47.37%)",
        "Observed Quantitative Impact": "Identical 47.37% reachability across all configurations; 0 false obstacles generated",
        "Scientific Conclusion / Scope": "Cost capping (< 240) strictly preserves reachable topology while obstacles remain permanently blocked (255)."
    }
]

df_table6 = pd.DataFrame(table6_data)
display(df_table6)
